# Ökobaudat Material Matcher

Automated pipeline that:
1. Decomposes a building element into material layers using Claude + web search + German/EU construction standards (DIN, EN ISO, GEG, DGNB)
2. Searches the Ökobaudat REST API for matching EPDs per layer
3. Ranks and selects the best EPD match per layer using Claude as LCA expert
4. Verifies the complete assembly against relevant German standards
5. Produces a structured JSON + human-readable report

**Run cells 1 → 2 → 3 → 4 in order.**

In [1]:
# Cell 1 — Install dependencies
!pip install anthropic requests rich -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.6/832.6 kB 8.0 MB/s eta 0:00:00


In [2]:
# Cell 2 — Set API key (Colab Secrets)
from google.colab import userdata
import os
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

In [5]:
# Cell 3 — Full pipeline code (with bug fix: handles name as str OR list)

import anthropic
import requests
import json
import time
import re
import os
from datetime import datetime
from typing import Any

try:
    from rich.console import Console
    from rich.panel import Panel
    from rich.table import Table
    from rich.markdown import Markdown
    from rich import print as rprint
    RICH = True
    console = Console()
except ImportError:
    RICH = False
    console = None

# ─────────────────────────────────────────────
# Ökobaudat API constants
# ─────────────────────────────────────────────
OEKOBAUDAT_BASE = "https://oekobaudat.de/OEKOBAU.DAT/resource"
DATASTOCK_UUID  = "cd2bda71-760b-4fcc-8a0b-3877c10000a8"   # OEKOBAUDAT 2021-II
DATASTOCK_NAME  = "OEKOBAUDAT 2021-II"
EPD_PAGE_SIZE   = 10

# ─────────────────────────────────────────────
# Claude client
# ─────────────────────────────────────────────
client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from env
MODEL  = "claude-opus-4-5"


# ══════════════════════════════════════════════════════════════════════════════
# Helpers
# ══════════════════════════════════════════════════════════════════════════════

def hdr(msg: str) -> None:
    if RICH: console.rule(f"[bold cyan]{msg}[/]")
    else: print(f"\n{'='*60}\n{msg}\n{'='*60}")

def info(msg: str) -> None:
    if RICH: console.print(f"  [dim]{msg}[/]")
    else: print(f"  {msg}")

def ok(msg: str) -> None:
    if RICH: console.print(f"  [green]✔[/] {msg}")
    else: print(f"  ✔ {msg}")

def warn(msg: str) -> None:
    if RICH: console.print(f"  [yellow]⚠[/]  {msg}")
    else: print(f"  ⚠  {msg}")

def err(msg: str) -> None:
    if RICH: console.print(f"  [red]✖[/] {msg}")
    else: print(f"  ✖ {msg}")

def extract_json(text: str) -> Any:
    """Extract the first JSON object or array from a possibly noisy string."""
    text = re.sub(r"```(?:json)?", "", text).strip()
    for start_char, end_char in [('{', '}'), ('[', ']')]:
        idx = text.find(start_char)
        if idx != -1:
            depth = 0
            for i, ch in enumerate(text[idx:], start=idx):
                if ch == start_char: depth += 1
                elif ch == end_char:
                    depth -= 1
                    if depth == 0:
                        candidate = text[idx:i+1]
                        try: return json.loads(candidate)
                        except json.JSONDecodeError: break
    raise ValueError(f"No valid JSON found in:\n{text[:400]}")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — Standards-based layer decomposition via web search
# ══════════════════════════════════════════════════════════════════════════════

def decompose_element(element: dict) -> dict:
    """
    Ask Claude to use web_search to find the relevant German/EU standards
    for this building element type, then decompose it into material layers.
    Returns:
      {
        "layers": [{"name", "thickness_mm", "description", "standard_ref", "search_keywords"}, ...],
        "references": [{"code", "title", "url"}, ...],
        "total_thickness_mm": int,
        "notes": str
      }
    """
    hdr("STEP 1 — Decompose element via web search + standards")
    info(f"Element: {element}")

    system = """You are a certified building physicist specialising in German construction standards.

Your task:
1. Use web_search to find the relevant German and EU norms for the given building element type.
   Search for: DIN 4108 (thermal insulation), DIN 18550, EN ISO 6946, GEG (Gebäudeenergiegesetz),
   DIN 18531 (flat roofs), DIN 18195 (waterproofing), DGNB criteria, and element-specific DIN norms.
2. Based on what you find, decompose the element into its standard material layers (inside-out order).
3. Ensure the sum of all layer thicknesses equals the given total thickness_mm (distribute proportionally
   if needed, but respect minimum code requirements per norm).

Output ONLY a single JSON object — no prose, no markdown fences — with exactly these keys:
{
  \"layers\": [
    {
      \"name\": \"concise 1-4 word material name suitable for database text search\",
      \"thickness_mm\": <integer>,
      \"description\": \"what this layer does and why it is required\",
      \"standard_ref\": \"norm that mandates or recommends this layer, e.g. DIN 4108-2\",
      \"search_keywords\": [\"alternative search term 1\", \"alternative search term 2\"]
    }
  ],
  \"references\": [
    {\"code\": \"DIN XXXX\", \"title\": \"full norm title\", \"url\": \"url if found\"}
  ],
  \"total_thickness_mm\": <sum of all layer thicknesses>,
  \"notes\": \"any important commentary on the decomposition\"
}"""

    user = f"""Building element definition:
{json.dumps(element, indent=2)}

Search for the relevant German construction standards for a {element.get('elementType', element.get('name', 'building element'))} and decompose it into standard material layers.
The total element thickness is {element.get('thickness', 'unspecified')} mm."""

    response = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        system=system,
        tools=[{"type": "web_search_20250305", "name": "web_search"}],
        messages=[{"role": "user", "content": user}]
    )

    text_parts = [b.text for b in response.content if b.type == "text"]
    raw_text = "\n".join(text_parts)
    result = extract_json(raw_text)

    layers = result.get("layers", [])
    refs   = result.get("references", [])
    ok(f"Decomposed into {len(layers)} layers")
    for i, layer in enumerate(layers, 1):
        info(f"  Layer {i}: {layer['name']} ({layer['thickness_mm']} mm) — {layer.get('standard_ref','')}")
    ok(f"Standards cited: {', '.join(r['code'] for r in refs)}")
    return result


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Search Ökobaudat REST API
# ══════════════════════════════════════════════════════════════════════════════

def search_oekobaudat(query: str, page_size: int = EPD_PAGE_SIZE) -> list[dict]:
    """
    Search OEKOBAUDAT 2021-II for processes matching the query name.
    API docs: https://www.oekobaudat.de/en/guidance/software-developers.html
    No authentication required — the API is publicly accessible.

    BUG FIX: The API can return 'name' as a plain string OR as a list of
    localised objects [{"lang": "en", "value": "..."}]. Both cases are handled.
    """
    url = (
        f"{OEKOBAUDAT_BASE}/datastocks/{DATASTOCK_UUID}/processes"
        f"?search=true&name={requests.utils.quote(query)}"
        f"&format=json&pageSize={page_size}&lang=en"
    )
    try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        data = r.json()
        results = []
        for p in (data.get("data") or []):
            # Handle name as string OR list (API is inconsistent)
            raw_name = p.get("name")
            if isinstance(raw_name, list):
                name_val = raw_name[0].get("value", p.get("uuid", "")) if raw_name else p.get("uuid", "")
            elif isinstance(raw_name, str):
                name_val = raw_name
            else:
                name_val = p.get("uuid", "")

            results.append({
                "uuid":    p.get("uuid", ""),
                "name":    name_val,
                "version": p.get("version", ""),
                "url":     f"https://www.oekobaudat.de/OEKOBAU.DAT/datasetdetail/process.xhtml?uuid={p.get('uuid','')}",
            })
        return results
    except requests.exceptions.ConnectionError as e:
        warn(f"    Network error for '{query}': {e}")
        return []
    except Exception as e:
        warn(f"    Ökobaudat query failed for '{query}': {e}")
        return []


def search_all_layers(layers: list[dict]) -> dict:
    """
    For each layer, try the primary name plus alternative search_keywords.
    Returns dict keyed by layer name -> list of candidate EPDs.
    """
    hdr("STEP 2 — Search Ökobaudat for each layer")
    all_results: dict[str, list] = {}

    for layer in layers:
        name = layer["name"]
        keywords = [name] + layer.get("search_keywords", [])
        seen_uuids: set = set()
        candidates: list = []

        for kw in keywords:
            hits = search_oekobaudat(kw)
            for h in hits:
                if h["uuid"] not in seen_uuids:
                    seen_uuids.add(h["uuid"])
                    candidates.append(h)
            if candidates:
                break

        # Final fallback: first single word
        if not candidates and " " in name:
            short = name.split()[0]
            for h in search_oekobaudat(short):
                if h["uuid"] not in seen_uuids:
                    seen_uuids.add(h["uuid"])
                    candidates.append(h)
            if candidates:
                info(f"    Fallback '{short}' → {len(candidates)} results")

        all_results[name] = candidates
        status = f"{len(candidates)} candidates" if candidates else "NO RESULTS"
        (ok if candidates else warn)(f"  '{name}' → {status}")

    return all_results


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Rank and select best EPD match per layer
# ══════════════════════════════════════════════════════════════════════════════

def rank_layer(layer: dict, candidates: list[dict]) -> dict:
    """Ask Claude to act as LCA expert and pick the best-matching EPD."""
    if not candidates:
        return {
            "best_uuid": None,
            "best_name": None,
            "confidence": "none",
            "reasoning": "No candidates found in Ökobaudat for this material.",
            "url": None,
        }

    system = """You are an LCA (Life Cycle Assessment) expert specialising in German construction materials.
Given a building layer description and a list of candidate EPDs from the Ökobaudat database,
select the single best-matching EPD.

Output ONLY a JSON object — no prose, no markdown fences:
{
  \"best_uuid\": \"<uuid of chosen EPD or null>\",
  \"best_name\": \"<name of chosen EPD or null>\",
  \"confidence\": \"high|medium|low|none\",
  \"reasoning\": \"1-2 sentence explanation of why this EPD was chosen\"
}"""

    user = f"""Layer to match:
{json.dumps(layer, indent=2)}

Candidate EPDs from Ökobaudat:
{json.dumps(candidates, indent=2)}

Select the best-matching EPD for this layer."""

    for attempt in range(3):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=500,
                system=system,
                messages=[{"role": "user", "content": user}]
            )
            raw = "".join(b.text for b in response.content if b.type == "text")
            result = extract_json(raw)
            if result.get("best_uuid"):
                result["url"] = f"https://www.oekobaudat.de/OEKOBAU.DAT/datasetdetail/process.xhtml?uuid={result['best_uuid']}"
            return result
        except Exception as e:
            if attempt < 2: time.sleep(1)
            else:
                err(f"Ranking failed for '{layer['name']}': {e}")
                return {"best_uuid": None, "best_name": None, "confidence": "none",
                        "reasoning": f"Ranking error: {e}", "url": None}


def rank_all_layers(layers: list[dict], search_results: dict) -> list[dict]:
    hdr("STEP 3 — Rank & select best EPD per layer")
    matched = []
    for layer in layers:
        name = layer["name"]
        candidates = search_results.get(name, [])
        info(f"  Ranking '{name}' ({len(candidates)} candidates)…")
        best = rank_layer(layer, candidates)
        conf_icon = {"high": "✔", "medium": "~", "low": "?", "none": "✖"}.get(best.get("confidence", "none"), "?")
        if best.get("best_name"):
            ok(f"  {conf_icon} '{name}' → {best['best_name']} [{best['confidence']}]")
        else:
            warn(f"  {conf_icon} '{name}' → no match")
        matched.append({"layer": layer, "best_match": best, "candidates": candidates})
    return matched


# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — Verify assembly against German standards
# ══════════════════════════════════════════════════════════════════════════════

def verify_against_standards(element: dict, decomposition: dict, matched: list[dict]) -> dict:
    hdr("STEP 4 — Verify assembly against German standards")

    system = """You are a certified German building physicist and code compliance expert.
Verify the given material assembly against relevant German construction standards.

Output ONLY a JSON object — no prose, no markdown fences:
{
  \"overall_verdict\": \"COMPLIANT|PARTIALLY_COMPLIANT|NON_COMPLIANT\",
  \"overall_score\": <0-100>,
  \"summary\": \"2-3 sentence executive summary\",
  \"layer_checks\": [
    {\"layer_name\": \"...\", \"status\": \"OK|WARNING|FAIL|NOT_MATCHED\", \"note\": \"...\"}
  ],
  \"missing_layers\": [\"layer description if mandatory layer is absent\"],
  \"recommendations\": [\"actionable improvement\"],
  \"standards_checked\": [\"DIN XXXX\", \"EN YYYY\"]
}"""

    assembly_summary = []
    for m in matched:
        assembly_summary.append({
            "layer_name":      m["layer"]["name"],
            "thickness_mm":    m["layer"]["thickness_mm"],
            "description":     m["layer"].get("description", ""),
            "standard_ref":    m["layer"].get("standard_ref", ""),
            "matched_epd":     m["best_match"].get("best_name"),
            "epd_uuid":        m["best_match"].get("best_uuid"),
            "match_confidence":m["best_match"].get("confidence"),
            "match_reasoning": m["best_match"].get("reasoning"),
        })

    user = f"""Building element:
{json.dumps(element, indent=2)}

Decomposition standards used:
{json.dumps(decomposition.get('references', []), indent=2)}

Matched material assembly (layer-by-layer):
{json.dumps(assembly_summary, indent=2)}

Decomposition notes: {decomposition.get('notes', '')}

Verify this assembly for compliance with German construction standards."""

    for attempt in range(3):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=1500,
                system=system,
                messages=[{"role": "user", "content": user}]
            )
            raw = "".join(b.text for b in response.content if b.type == "text")
            result = extract_json(raw)
            verdict = result.get("overall_verdict", "UNKNOWN")
            score   = result.get("overall_score", 0)
            color   = "green" if verdict == "COMPLIANT" else "yellow" if "PARTIAL" in verdict else "red"
            if RICH: console.print(f"\n  [bold {color}]Verdict: {verdict} ({score}/100)[/]")
            else: print(f"\n  Verdict: {verdict} ({score}/100)")
            return result
        except Exception as e:
            if attempt < 2: time.sleep(1)
            else:
                err(f"Verification failed: {e}")
                return {"overall_verdict": "ERROR", "summary": str(e)}


# ══════════════════════════════════════════════════════════════════════════════
# Report rendering
# ══════════════════════════════════════════════════════════════════════════════

def render_report(element: dict, decomposition: dict, matched: list[dict], verification: dict) -> str:
    lines = [
        "# Ökobaudat Material Matching Report",
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
        f"Datastock: {DATASTOCK_NAME}", "",
        "## Building Element",
        f"- **Name:** {element.get('name')}",
        f"- **Type:** {element.get('elementType', '—')}",
        f"- **Total thickness:** {element.get('thickness')} mm", "",
        "## Layer Decomposition (inside → outside)", "",
    ]
    for i, m in enumerate(matched, 1):
        layer = m["layer"]
        best  = m["best_match"]
        conf  = best.get("confidence", "none")
        conf_badge = {"high": "🟢 HIGH", "medium": "🟡 MEDIUM", "low": "🟠 LOW", "none": "🔴 NO MATCH"}.get(conf, conf)
        lines += [
            f"### Layer {i}: {layer['name']} ({layer['thickness_mm']} mm)",
            f"- **Description:** {layer.get('description', '—')}",
            f"- **Standard ref:** {layer.get('standard_ref', '—')}",
            f"- **EPD match:** {best.get('best_name', 'None')}  {conf_badge}",
        ]
        if best.get("best_uuid"):
            lines.append(f"- **UUID:** `{best['best_uuid']}`")
            lines.append(f"- **Link:** {best.get('url', '')}")
        lines.append(f"- **Reasoning:** {best.get('reasoning', '—')}")
        lines.append(f"- **Candidates searched:** {len(m['candidates'])}")
        lines.append("")

    refs = decomposition.get("references", [])
    if refs:
        lines += ["## Standards & References", ""]
        for r in refs:
            url_part = f" — {r.get('url', '')}" if r.get("url") else ""
            lines.append(f"- **{r['code']}**: {r.get('title', '')}{url_part}")
        lines.append("")

    v = verification
    lines += [
        "## Standards Compliance Verification", "",
        f"**Overall verdict:** {v.get('overall_verdict', '—')}  |  **Score:** {v.get('overall_score', '—')}/100",
        "", f"{v.get('summary', '')}", "",
    ]
    checks = v.get("layer_checks", [])
    if checks:
        lines += ["### Layer-by-layer checks", ""]
        for c in checks:
            icon = {"OK": "✅", "WARNING": "⚠️", "FAIL": "❌", "NOT_MATCHED": "❓"}.get(c.get("status", ""), "•")
            lines.append(f"- {icon} **{c.get('layer_name')}** — {c.get('note', '')}")
        lines.append("")
    missing = v.get("missing_layers", [])
    if missing:
        lines += ["### Missing mandatory layers", ""]
        for ml in missing: lines.append(f"- ⚠️ {ml}")
        lines.append("")
    recs = v.get("recommendations", [])
    if recs:
        lines += ["### Recommendations", ""]
        for rec in recs: lines.append(f"- {rec}")
        lines.append("")
    checked = v.get("standards_checked", [])
    if checked:
        lines.append(f"**Standards checked:** {', '.join(checked)}")
        lines.append("")
    return "\n".join(lines)


# ══════════════════════════════════════════════════════════════════════════════
# Main pipeline
# ══════════════════════════════════════════════════════════════════════════════

def run_pipeline(element: dict, output_path: str | None = None) -> dict:
    if RICH:
        console.print(Panel(
            f"[bold]Ökobaudat Material Matching Pipeline[/]\n"
            f"Element: [cyan]{element.get('name')}[/]  |  "
            f"Type: {element.get('elementType', '—')}  |  "
            f"Thickness: {element.get('thickness')} mm",
            border_style="cyan"
        ))
    else:
        print(f"\n{'═'*60}\n Ökobaudat Material Matching Pipeline\n Element: {element}\n{'═'*60}")

    decomposition  = decompose_element(element)
    layers         = decomposition["layers"]
    search_results = search_all_layers(layers)
    matched        = rank_all_layers(layers, search_results)
    verification   = verify_against_standards(element, decomposition, matched)
    report_md      = render_report(element, decomposition, matched, verification)

    if RICH:
        console.print()
        console.print(Markdown(report_md))
    else:
        print("\n" + report_md)

    output = {
        "element":         element,
        "generated_at":    datetime.now().isoformat(),
        "datastock":       DATASTOCK_NAME,
        "decomposition":   decomposition,
        "matched_layers":  [{"layer": m["layer"], "best_match": m["best_match"], "candidates": m["candidates"]} for m in matched],
        "verification":    verification,
        "report_markdown": report_md,
    }

    if output_path:
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(output, f, indent=2, ensure_ascii=False)
        ok(f"\nResults saved to: {output_path}")

    return output


def main(element: dict = None, output_path: str = "okobaudat_result.json") -> dict:
    """
    Call this function from Jupyter/Colab:
        result = main({"name":"External Wall", "elementType":"Wall", "thickness":300})
    """
    DEFAULT_ELEMENT = {
        "name":        "Generic Roof",
        "elementType": "Roof",
        "thickness":   200,
    }
    return run_pipeline(element or DEFAULT_ELEMENT, output_path=output_path)


print("✔ Pipeline loaded. Edit the element in Cell 4 and run it.")

✔ Pipeline loaded. Edit the element in Cell 4 and run it.


In [6]:
# Cell 4 — Run the pipeline
# Edit the element definition below, then run this cell.

element = {
    "name":        "Generic Roof",
    "elementType": "Roof",
    "thickness":   200,
}

# Other examples to try:
# element = {"name": "External Wall",  "elementType": "Wall",  "thickness": 300}
# element = {"name": "Floor Slab",     "elementType": "Floor", "thickness": 250}
# element = {"name": "Basement Wall",  "elementType": "Wall",  "thickness": 350}

result = main(element, output_path="okobaudat_result.json")

# Download the JSON result
try:
    from google.colab import files
    files.download("okobaudat_result.json")
except ImportError:
    print("Not in Colab — result saved to okobaudat_result.json")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Ökobaudat Material Matching Pipeline                                                                            │
│ Element: Generic Roof  |  Type: Roof  |  Thickness: 200 mm                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

────────────────────────────── STEP 1 — Decompose element via web search + standards ──────────────────────────────

Element: {'name': 'Generic Roof', 'elementType': 'Roof', 'thickness': 200}

✔ Decomposed into 5 layers

  Layer 1: Reinforced Concrete Deck (100 mm) — DIN 1045, DIN 18531-1

...

Verdict: PARTIALLY_COMPLIANT (62/100)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                       Ökobaudat Material Matching Report                                        ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

Generated: 2026-05-21 05:39 Datastock: OEKOBAUDAT 2021-II                                                          


                                                 Building Element                                                  

 • Name: Generic Roof                                                                                              
 • Type: Roof                                                                                                      
 • Total thickness: 200 mm                                                                                         


                                      Layer Decomposition (inside → outside)                                       

                                    Layer 1: Reinforced Concrete Deck (100 mm)                                     

 • Description: Structural supporting layer providing load-bearing capacity for the roof assembly. Forms the       
   substrate for the warm roof construction system.                                                                
 • Standard ref: DIN 1045, DIN 18531-1                                                                             
 • EPD match: Precast concrete slab, ceiling, thickness 20cm; 504 kg/m2  🟡 MEDIUM                                 
 • UUID: 6575f9dd-8a50-440c-90df-30608167c739                                                                      
 • Link:                                                                                                           
   https://www.oekobaudat.de/OEKOBAU.DAT/datasetdetail/process.xhtml?uuid=6575f9dd-8a50-440c-90df-30608167c739     
 • Reasoning: This is the most recent version (20.21.060) of a ceiling/deck concrete slab EPD. While the layer     
   specifies 100mm thickness and this EPD is for 200mm, it's the closest ceiling application match available. The  
   20cm slab can be scaled proportionally for the 10cm layer, and ceiling slabs are appropriate for roof deck      
   applications per DIN 18531-1.                                                                                   
 • Candidates searched: 5                                                                                          

                                      Layer 2: Bitumen Vapour Barrier (5 mm)                                       

 • Description: Diffusion-tight layer preventing water vapour from indoor air penetrating into the insulation layer
   and causing condensation damage. Required with sd-value ≥ 1500m for flat roof constructions where external      
   waterproofing is vapour-tight.                                                                                  
 • Standard ref: DIN 4108-3, DIN EN 13970                                                                          
 • EPD match: DuPont™ AirGuard® Air & Vapour Control Layers (5816X)  🟡 MEDIUM                                     
 • UUID: 9d445672-95f5-4273-9147-1986fd2f5a96                                                                      
 • Link:                                                                                                           
   https://www.oekobaudat.de/OEKOBAU.DAT/datasetdetail/process.xhtml?uuid=9d445672-95f5-4273-9147-1986fd2f5a96     
 • Reasoning: This EPD is for a vapour control layer which matches the functional requirement of the bitumen vapour
   barrier. However, the layer description specifically calls for a bitumen membrane while the DuPont AirGuard is a
   synthetic membrane system, so material composition differs though function aligns.                              
 • Candidates searched: 1                       

✔ 
Results saved to: okobaudat_result.json

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>